In [ ]:
import pandas as pd
import os
import glob
from shapely.geometry import LineString, Point

# Path to directory with parquet files
dir_path = "scratch/Oulu_GTFS_wCO2"

# Find all .parquet files
parquet_files = glob.glob(os.path.join(dir_path, "**/*.parquet"), recursive=True)

# Check if any files were found
if not parquet_files:
    raise FileNotFoundError("No .parquet files found in the specified directory.")

# Read only the first 400 parquet files
df_list = [pd.read_parquet(fp) for fp in parquet_files]
df_all = pd.concat(df_list, ignore_index=True)

# Display basic info
print(f"Loaded {len(df_list)} parquet files.")


In [ ]:
df_all.shape

In [ ]:
def calculate_co2_walking(row):
    modes = row['pt_trip_seq'].tolist()   # convert to list
    distances = row['pt_dist_seq'].tolist()
    
    total_co2 = 0
    for mode, dist in zip(modes, distances):
        if mode == 'WALK':
            total_co2 += dist * 0.056 #EMISSION_FACTOR_WALK
    return total_co2

df_all['co2_walking'] = df_all.apply(calculate_co2_walking, axis=1)

In [ ]:
df_all

In [ ]:
import geopandas as gpd

# Load the study area polygon
gdf = gpd.read_file("./data/oulu_region_boundary.geojson")  # or .shp

# Make sure it’s in WGS84 (lat/lon) for OSMnx
gdf = gdf.to_crs(epsg=4326)


In [ ]:
import osmnx as ox
import geopandas as gpd


# Extract the geometry
polygon = gdf.unary_union  # or gdf.geometry.iloc[0] if only one feature

default_access = '["access"!~"private"]'  # example placeholder

# Custom OSM filter for the cycling network
custom_bike_filter = (
    f'["highway"]["area"!~"yes"]{default_access}'
    f'["highway"!~"abandoned|bus_guideway|corridor|elevator|'
    f'escalator|motor|no|planned|platform|proposed|raceway|razed|'
    f'rest_area|services|steps"]'
    f'["service"!~"private"]'
    f'["bicycle"!~"no"]'
)

# Download the bike network within the polygon
G = ox.graph_from_polygon(polygon, custom_filter=custom_bike_filter, network_type=None)

In [ ]:
G = ox.truncate.largest_component(G, strongly=False)

In [ ]:
node_coords = {node: (data["x"], data["y"]) for node, data in G.nodes(data=True)}

In [ ]:
travel_time_matrix = pd.read_parquet("./data/OD_cycling_33M_snap_1_oulu.parquet")

In [ ]:
# Build a GeoDataFrame with the projected nodes
nodes_gdf = gpd.GeoDataFrame(
    {"node": list(node_coords.keys())},
    geometry=[Point(xy) for xy in node_coords.values()],
    crs=G.graph["crs"]  # graph CRS
)

# Reproyectar a WGS84
nodes_wgs84 = nodes_gdf.to_crs(epsg=4326)

# Build a new dict {node: (lon, lat)}
node_coords_wgs84 = dict(zip(
    nodes_wgs84["node"],
    [(geom.x, geom.y) for geom in nodes_wgs84.geometry]
))

In [ ]:

# assuming travel_time_matrix has columns: orig_node, dest_node, from_lon, from_lat, to_lon, to_lat
# node_coords_wgs84 is a dict: {node_id: (lon, lat)}

# filter orig nodes that exist
valid_orig_nodes = [n for n in travel_time_matrix["orig_node"] if n in node_coords_wgs84]
valid_dest_nodes = [n for n in travel_time_matrix["dest_node"] if n in node_coords_wgs84]

# create DataFrame with coordinates for valid orig nodes
df_orig_nodes = pd.DataFrame({
    "node": valid_orig_nodes,
    "lon": [node_coords_wgs84[n][0] for n in valid_orig_nodes],
    "lat": [node_coords_wgs84[n][1] for n in valid_orig_nodes]
})

# create DataFrame with coordinates for valid dest nodes
df_dest_nodes = pd.DataFrame({
    "node": valid_dest_nodes,
    "lon": [node_coords_wgs84[n][0] for n in valid_dest_nodes],
    "lat": [node_coords_wgs84[n][1] for n in valid_dest_nodes]
})

# combine them (optional, remove duplicates)
df_nodes_all = pd.concat([df_orig_nodes, df_dest_nodes]).drop_duplicates(subset="node").reset_index(drop=True)


In [ ]:
import numpy as np
from pyproj import Geod

geod = Geod(ellps="WGS84")

# helper function to get node coordinates safely
def safe_coords(n):
    if n in node_coords_wgs84:
        return node_coords_wgs84[n]
    else:
        return (np.nan, np.nan)  # placeholder to compute a default distance later

# get origin coordinates safely
orig_coords = np.array([safe_coords(n) for n in travel_time_matrix["orig_node"]])
dest_coords = np.array([safe_coords(n) for n in travel_time_matrix["dest_node"]])

# compute distances
_, _, dist_orig = geod.inv(
    travel_time_matrix["from_lon"].to_numpy(),
    travel_time_matrix["from_lat"].to_numpy(),
    orig_coords[:,0],
    orig_coords[:,1]
)

_, _, dist_dest = geod.inv(
    travel_time_matrix["to_lon"].to_numpy(),
    travel_time_matrix["to_lat"].to_numpy(),
    dest_coords[:,0],
    dest_coords[:,1]
)

# replace distances where coordinates were missing with 5000
dist_orig = np.where(np.isnan(orig_coords[:,0]), 5000, dist_orig)
dist_dest = np.where(np.isnan(dest_coords[:,0]), 5000, dist_dest)


In [ ]:
# Store in the DataFrame
travel_time_matrix["dist_orig_m"] = dist_orig
travel_time_matrix["dist_dest_m"] = dist_dest

In [ ]:
travel_time_matrix.to_parquet("./data/travel_time_node_distance_islands_oulu.parquet")

In [ ]:
travel_time_matrix = pd.read_parquet("./data/travel_time_node_distance_islands_oulu.parquet")

In [ ]:
# Definir un umbral (ejemplo: 1000 m)
threshold = 500  

# Detect problematic rows
mask_bad = (travel_time_matrix["dist_orig_m"] > threshold) | (travel_time_matrix["dist_dest_m"] > threshold)

# Subset of problematic rows
bad_matches = travel_time_matrix[mask_bad]

print(f"Warning: {mask_bad.sum()} rows with distances greater than {threshold} m")

In [ ]:

threshold = 500  
# 1) Get unique bad from_ids and to_ids separately
bad_from_ids = bad_matches.loc[bad_matches["dist_orig_m"] > threshold, "from_id"].unique()
bad_to_ids   = bad_matches.loc[bad_matches["dist_dest_m"] > threshold, "to_id"].unique()

# 2) Filter df_all
filtered_result = df_all[
    ~df_all["from_id"].isin(bad_from_ids) &
    ~df_all["to_id"].isin(bad_to_ids)
].copy()

print(f"Removed {len(df_all) - len(filtered_result)} rows from df_all")


In [ ]:
df_all = filtered_result.copy()

In [ ]:
df_all["pt_co2_total"] = df_all["pt_co2"] + df_all["co2_walking"]

In [ ]:
df_pt_co2_3000 = df_all[df_all["pt_co2_total"] <= 3000].copy()

In [ ]:
df_pt_co2_3000.to_parquet("./output/pt_co2_3000_oulu.parquet")

In [ ]:
df_pt_co2_3000

In [ ]:
# Filter rows where pt_time is less than or equal to 15, 30, and 45
df_pt_15 = df_all[df_all["pt_time"] <= 15].copy()
df_pt_30 = df_all[df_all["pt_time"] <= 30].copy()
df_pt_45 = df_all[df_all["pt_time"] <= 45].copy()

In [ ]:
df_all["pt_co2_total"] = df_all["pt_co2"] + df_all["co2_walking"]

In [ ]:
# Filter rows where pt_time is less than or equal to 15, 30, and 45
df_pt_co2_125 = df_all[df_all["pt_co2_total"] <= 125].copy()
df_pt_co2_250 = df_all[df_all["pt_co2_total"] <= 250].copy()
df_pt_co2_05 = df_all[df_all["pt_co2_total"] <= 500].copy()

In [ ]:
# Filter rows where pt_time is less than or equal to 15, 30, and 45
df_pt_co2_2000 = df_all[df_all["pt_co2_total"] <= 2000].copy()

In [ ]:
hsk_pois = pd.read_parquet("./data/pois_per_hex.parquet")

In [ ]:
hsk_pois

In [ ]:
df_pt_co2_2000

In [ ]:
# Step 1: Group hsk_pois by 'h3_id' and 'category' to get counts
pois_grouped = (
    hsk_pois
    .groupby(['h3_id', 'category'])['count']
    .sum()                           # sum the values instead of counting rows
    .unstack(fill_value=0)           # make wide format, categories as columns
    .reset_index()
)

# Step 2: Merge with df_pt_15 using 'to_id' (in df_pt_15) and 'h3_id' (in pois_grouped)
df_pt_co2_2000_merged = df_pt_co2_2000.merge(pois_grouped, how='left', left_on='to_id', right_on='h3_id')
df_pt_co2_2000_merged = df_pt_co2_2000_merged.drop(columns=['h3_id'])



# Step 4: Add category columns from origin and destination
category_cols = pois_grouped.columns.drop('h3_id')
for col in category_cols:
   df_pt_co2_2000_merged[col] = df_pt_co2_2000_merged[col].fillna(0) + 0

In [ ]:
df_pt_co2_2000_merged.to_parquet("./output/pt_co2_2000.parquet")

In [ ]:
df_pt_co2_2000_merged.sort_values(by="Jobs, Professional Services & Religious")

In [ ]:


# Step 2: Merge with df_pt_15 using 'to_id' (in df_pt_15) and 'h3_id' (in pois_grouped)
df_pt_co2_250_merged = df_pt_co2_250.merge(pois_grouped, how='left', left_on='to_id', right_on='h3_id')
#df_pt_co2_250_merged = df_pt_co2_250.drop(columns=['h3_id'])


# Step 4: Add category columns from origin and destination
category_cols = pois_grouped.columns.drop('h3_id')
for col in category_cols:
    df_pt_co2_250_merged[col] = df_pt_co2_250_merged[col].fillna(0) + 0

In [ ]:
# Step 5: Group by 'from_id', sum POI categories, and average pt_co2
category_cols = [
    'Educational Facilities',
    'Grocery Stores & Supermarkets',
    'Jobs, Professional Services & Religious',
    'Restaurant & Entertainment',
    'Shopping & Retail',
    'Uncategorized',
    'Well-being & Lifestyle'
]

grouped_summary = df_pt_co2_250_merged.groupby('from_id')[category_cols + ['pt_time']].agg({
    'Educational Facilities': 'sum',
    'Grocery Stores & Supermarkets': 'sum',
    'Jobs, Professional Services & Religious': 'sum',
    'Restaurant & Entertainment': 'sum',
    'Shopping & Retail': 'sum',
    'Uncategorized': 'sum',
    'Well-being & Lifestyle': 'sum',
    'pt_time':'mean'
    
}).reset_index()


In [ ]:
grouped_summary

In [ ]:
# Step 6: Create total_pois column (excluding 'Uncategorized')
grouped_summary['total_pois'] = grouped_summary[[
    'Educational Facilities',
    'Grocery Stores & Supermarkets',
    'Jobs, Professional Services & Religious',
    'Restaurant & Entertainment',
    'Shopping & Retail',
    'Well-being & Lifestyle'
]].sum(axis=1)

In [ ]:
import geopandas as gpd
import h3
from shapely.geometry import Polygon
import contextily as ctx; import basemaps
import matplotlib.pyplot as plt
import mapclassify
import matplotlib.patches as mpatches

# Function to convert h3 to polygon
def h3_to_polygon(h):
    return Polygon(h3.h3_to_geo_boundary(h, geo_json=True))

# Convert 'from_id' to geometry
grouped_summary = grouped_summary.copy()
geometry = grouped_summary['from_id'].apply(h3_to_polygon)
gdf_hexes = gpd.GeoDataFrame(grouped_summary, geometry=geometry, crs='EPSG:4326')

# Classify total_pois into 5 natural breaks
classifier = mapclassify.NaturalBreaks(y=gdf_hexes['total_pois'], k=5)
gdf_hexes['poi_class'] = classifier.yb

# Get bin edges for legend
bin_edges = classifier.bins

# Define custom legend handles
legend_handles = []
for i in range(len(bin_edges)):
    if i == 0:
        label = f"<= {bin_edges[i]:.1f}"
    else:
        label = f"> {bin_edges[i-1]:.1f} – {bin_edges[i]:.1f}"
    patch = mpatches.Patch(color=plt.cm.viridis(i / (len(bin_edges)-1)), label=label)
    legend_handles.append(patch)

# Plot
fig, ax = plt.subplots(figsize=(10, 10))
gdf_hexes.plot(ax=ax, column='poi_class', cmap='viridis', legend=False, edgecolor='black', linewidth=0.2)
ctx.add_basemap(ax, source=basemaps.POSITRON, crs=gdf_hexes.crs.to_string())
ax.set_title("Total POIs per Hexagon (Natural Breaks)")
ax.axis('off')
plt.legend(handles=legend_handles, title="Total POIs", loc='lower left')
plt.tight_layout()
plt.show()

In [ ]:
# Step 1: Group hsk_pois by 'h3_id' and 'category' to get counts
pois_grouped = hsk_pois.groupby(['h3_id', 'category']).size().unstack(fill_value=0).reset_index()

# Step 2: Merge with df_pt_15 using 'to_id' (in df_pt_15) and 'h3_id' (in pois_grouped)
df_pt_co2_05_merged = df_pt_co2_05.merge(pois_grouped, how='left', left_on='to_id', right_on='h3_id')
df_pt_co2_05_merged = df_pt_co2_05_merged.drop(columns=['h3_id'])

# Step 3: Merge again using 'from_id' to get POIs at origin
from_pois = df_pt_co2_05.merge(pois_grouped, how='left', left_on='from_id', right_on='h3_id')
from_pois = from_pois.drop(columns=['h3_id'])

# Step 4: Add category columns from origin and destination
category_cols = pois_grouped.columns.drop('h3_id')
for col in category_cols:
    df_pt_co2_05_merged[col] = df_pt_co2_05_merged[col].fillna(0) + from_pois[col].fillna(0)

In [ ]:
category_cols = [
    'Educational Facilities',
    'Grocery Stores & Supermarkets',
    'Jobs, Professional Services & Religious',
    'Restaurant & Entertainment',
    'Shopping & Retail',
    'Uncategorized',
    'Well-being & Lifestyle'
]

grouped_summary_05 = df_pt_co2_05_merged.groupby('from_id')[category_cols + ['pt_time']].agg({
    'Educational Facilities': 'sum',
    'Grocery Stores & Supermarkets': 'sum',
    'Jobs, Professional Services & Religious': 'sum',
    'Restaurant & Entertainment': 'sum',
    'Shopping & Retail': 'sum',
    'Uncategorized': 'sum',
    'Well-being & Lifestyle': 'sum',
    'pt_time':'mean'
    
}).reset_index()


In [ ]:
# Step 6: Create total_pois column (excluding 'Uncategorized')
grouped_summary_05['total_pois'] = grouped_summary_05[[
    'Educational Facilities',
    'Grocery Stores & Supermarkets',
    'Jobs, Professional Services & Religious',
    'Restaurant & Entertainment',
    'Shopping & Retail',
    'Well-being & Lifestyle'
]].sum(axis=1)

In [ ]:
import geopandas as gpd
import h3
from shapely.geometry import Polygon
import contextily as ctx; import basemaps
import matplotlib.pyplot as plt
import mapclassify
import matplotlib.patches as mpatches

# Function to convert h3 to polygon
def h3_to_polygon(h):
    return Polygon(h3.h3_to_geo_boundary(h, geo_json=True))

# Convert 'from_id' to geometry
grouped_summary_05 = grouped_summary_05.copy()
geometry = grouped_summary_05['from_id'].apply(h3_to_polygon)
gdf_hexes = gpd.GeoDataFrame(grouped_summary_05, geometry=geometry, crs='EPSG:4326')

# Classify total_pois into 5 natural breaks
classifier = mapclassify.NaturalBreaks(y=gdf_hexes['total_pois'], k=5)
gdf_hexes['poi_class'] = classifier.yb

# Get bin edges for legend
bin_edges = classifier.bins

# Define custom legend handles
legend_handles = []
for i in range(len(bin_edges)):
    if i == 0:
        label = f"<= {bin_edges[i]:.1f}"
    else:
        label = f"> {bin_edges[i-1]:.1f} – {bin_edges[i]:.1f}"
    patch = mpatches.Patch(color=plt.cm.viridis(i / (len(bin_edges)-1)), label=label)
    legend_handles.append(patch)

# Plot
fig, ax = plt.subplots(figsize=(10, 10))
gdf_hexes.plot(ax=ax, column='poi_class', cmap='viridis', legend=False, edgecolor='black', linewidth=0.2)
ctx.add_basemap(ax, source=basemaps.POSITRON, crs=gdf_hexes.crs.to_string())
ax.set_title("Total POIs per Hexagon (Natural Breaks)")
ax.axis('off')
plt.legend(handles=legend_handles, title="Total POIs", loc='lower left')
plt.tight_layout()
plt.show()


In [ ]:
import geopandas as gpd
import h3
from shapely.geometry import Polygon
import contextily as ctx; import basemaps
import matplotlib.pyplot as plt
import mapclassify
import matplotlib.patches as mpatches

# Function to convert h3 index to shapely Polygon
def h3_to_polygon(h):
    return Polygon(h3.h3_to_geo_boundary(h, geo_json=True))

# Define POI categories to plot
poi_categories = [
    'Educational Facilities',
    'Grocery Stores & Supermarkets',
    'Jobs, Professional Services & Religious',
    'Restaurant & Entertainment',
    'Shopping & Retail',
    'Well-being & Lifestyle'
]

# Create GeoDataFrame with geometry if not already created
if 'geometry' not in grouped_summary.columns:
    geometry = grouped_summary['from_id'].apply(h3_to_polygon)
    gdf_hexes = gpd.GeoDataFrame(grouped_summary, geometry=geometry, crs='EPSG:4326')
else:
    gdf_hexes = grouped_summary.copy()

# Set up plot grid
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

# Generate map for each category
for idx, category in enumerate(poi_categories):
    ax = axes[idx]
    # Classify with Natural Breaks
    classifier = mapclassify.NaturalBreaks(y=gdf_hexes[category], k=5)
    gdf_hexes['poi_class'] = classifier.yb
    bin_edges = classifier.bins

    # Create custom legend
    legend_handles = []
    for i in range(len(bin_edges)):
        if i == 0:
            label = f"<= {bin_edges[i]:.1f}"
        else:
            label = f"> {bin_edges[i-1]:.1f} – {bin_edges[i]:.1f}"
        patch = mpatches.Patch(color=plt.cm.viridis(i / (len(bin_edges) - 1)), label=label)
        legend_handles.append(patch)

    # Plot
    gdf_hexes.plot(ax=ax, column='poi_class', cmap='viridis', legend=False, edgecolor='black', linewidth=0.2)
    ctx.add_basemap(ax, source=basemaps.POSITRON, crs=gdf_hexes.crs.to_string())
    ax.set_title(category)
    ax.axis('off')
    ax.legend(handles=legend_handles, title=category, loc='upper left')

plt.suptitle("POI Categories by Hexagon ina 500g trip (Natural Breaks)", fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()


In [ ]:
# Step 1: Group hsk_pois by 'h3_id' and 'category' to get counts
pois_grouped = hsk_pois.groupby(['h3_id', 'category']).size().unstack(fill_value=0).reset_index()

# Step 2: Merge with df_pt_15 using 'to_id' (in df_pt_15) and 'h3_id' (in pois_grouped)
df_pt_co2_250_merged = df_pt_co2_250.merge(pois_grouped, how='left', left_on='to_id', right_on='h3_id')
df_pt_co2_250_merged = df_pt_co2_250_merged.drop(columns=['h3_id'])

# Step 3: Merge again using 'from_id' to get POIs at origin
from_pois = df_pt_co2_250.merge(pois_grouped, how='left', left_on='from_id', right_on='h3_id')
from_pois = from_pois.drop(columns=['h3_id'])

# Step 4: Add category columns from origin and destination
category_cols = pois_grouped.columns.drop('h3_id')
for col in category_cols:
    df_pt_co2_250_merged[col] = df_pt_co2_250_merged[col].fillna(0) + from_pois[col].fillna(0)

In [ ]:
category_cols = [
    'Educational Facilities',
    'Grocery Stores & Supermarkets',
    'Jobs, Professional Services & Religious',
    'Restaurant & Entertainment',
    'Shopping & Retail',
    'Uncategorized',
    'Well-being & Lifestyle'
]

grouped_summary_250 = df_pt_co2_250_merged.groupby('from_id')[category_cols + ['pt_time']].agg({
    'Educational Facilities': 'sum',
    'Grocery Stores & Supermarkets': 'sum',
    'Jobs, Professional Services & Religious': 'sum',
    'Restaurant & Entertainment': 'sum',
    'Shopping & Retail': 'sum',
    'Uncategorized': 'sum',
    'Well-being & Lifestyle': 'sum',
    'pt_time':'mean'
    
}).reset_index()


In [ ]:
# Step 6: Create total_pois column (excluding 'Uncategorized')
grouped_summary_250['total_pois'] = grouped_summary_250[[
    'Educational Facilities',
    'Grocery Stores & Supermarkets',
    'Jobs, Professional Services & Religious',
    'Restaurant & Entertainment',
    'Shopping & Retail',
    'Well-being & Lifestyle'
]].sum(axis=1)

In [ ]:
# Function to convert h3 to polygon
def h3_to_polygon(h):
    return Polygon(h3.h3_to_geo_boundary(h, geo_json=True))

# Convert 'from_id' to geometry
grouped_summary_250 = grouped_summary_250.copy()
geometry = grouped_summary_250['from_id'].apply(h3_to_polygon)
gdf_hexes = gpd.GeoDataFrame(grouped_summary_250, geometry=geometry, crs='EPSG:4326')

# Classify total_pois into 5 natural breaks
classifier = mapclassify.NaturalBreaks(y=gdf_hexes['total_pois'], k=5)
gdf_hexes['poi_class'] = classifier.yb

# Get bin edges for legend
bin_edges = classifier.bins

# Define custom legend handles
legend_handles = []
for i in range(len(bin_edges)):
    if i == 0:
        label = f"<= {bin_edges[i]:.1f}"
    else:
        label = f"> {bin_edges[i-1]:.1f} – {bin_edges[i]:.1f}"
    patch = mpatches.Patch(color=plt.cm.viridis(i / (len(bin_edges)-1)), label=label)
    legend_handles.append(patch)

# Plot
fig, ax = plt.subplots(figsize=(10, 10))
gdf_hexes.plot(ax=ax, column='poi_class', cmap='viridis', legend=False, edgecolor='black', linewidth=0.2)
ctx.add_basemap(ax, source=basemaps.POSITRON, crs=gdf_hexes.crs.to_string())
ax.set_title("Total POIs per Hexagon (Natural Breaks)")
ax.axis('off')
plt.legend(handles=legend_handles, title="Total POIs", loc='lower left')
plt.tight_layout()
plt.show()

In [ ]:
import geopandas as gpd
import h3
from shapely.geometry import Polygon
import contextily as ctx; import basemaps
import matplotlib.pyplot as plt
import mapclassify
import matplotlib.patches as mpatches

# Function to convert h3 to polygon
def h3_to_polygon(h):
    return Polygon(h3.h3_to_geo_boundary(h, geo_json=True))

# Convert 'from_id' to geometry
grouped_summary_250 = grouped_summary_250.copy()
geometry = grouped_summary_250['from_id'].apply(h3_to_polygon)
gdf_hexes = gpd.GeoDataFrame(grouped_summary_250, geometry=geometry, crs='EPSG:4326')

# Define POI categories
poi_categories = [
    'Educational Facilities',
    'Grocery Stores & Supermarkets',
    'Jobs, Professional Services & Religious',
    'Restaurant & Entertainment',
    'Shopping & Retail',
    'Well-being & Lifestyle'
]

# Set up figure and axes
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(18, 12))
axes = axes.flatten()

for i, category in enumerate(poi_categories):
    ax = axes[i]
    
    # Classify with natural breaks
    classifier = mapclassify.NaturalBreaks(y=gdf_hexes[category], k=5)
    gdf_hexes[f'{category}_class'] = classifier.yb
    bin_edges = classifier.bins

    # Create custom legend
    legend_handles = []
    for j in range(len(bin_edges)):
        if j == 0:
            label = f"<= {bin_edges[j]:.1f}"
        else:
            label = f"> {bin_edges[j-1]:.1f} – {bin_edges[j]:.1f}"
        patch = mpatches.Patch(color=plt.cm.viridis(j / (len(bin_edges)-1)), label=label)
        legend_handles.append(patch)

    # Plot
    gdf_hexes.plot(ax=ax, column=f'{category}_class', cmap='viridis', legend=False, edgecolor='black', linewidth=0.2)
    ctx.add_basemap(ax, source=basemaps.POSITRON, crs=gdf_hexes.crs.to_string())
    ax.set_title(category)
    ax.axis('off')
    ax.legend(handles=legend_handles, title=category, loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:

# Step 1: Group hsk_pois by 'h3_id' and 'category' to get counts
pois_grouped = hsk_pois.groupby(['h3_id', 'category']).size().unstack(fill_value=0).reset_index()

# Step 2: Merge with df_pt_15 using 'to_id' (in df_pt_15) and 'h3_id' (in pois_grouped)
df_pt_15_merged = df_pt_15.merge(pois_grouped, how='left', left_on='to_id', right_on='h3_id')
df_pt_15_merged = df_pt_15_merged.drop(columns=['h3_id'])

# Step 3: Merge again using 'from_id' to get POIs at origin
from_pois = df_pt_15.merge(pois_grouped, how='left', left_on='from_id', right_on='h3_id')
from_pois = from_pois.drop(columns=['h3_id'])

# Step 4: Add category columns from origin and destination
category_cols = pois_grouped.columns.drop('h3_id')
for col in category_cols:
    df_pt_15_merged[col] = df_pt_15_merged[col].fillna(0) + from_pois[col].fillna(0)


In [ ]:
df_pt_15_merged.head()

In [ ]:
# Step 5: Group by 'from_id', sum POI categories, and average pt_co2
category_cols = [
    'Educational Facilities',
    'Grocery Stores & Supermarkets',
    'Jobs, Professional Services & Religious',
    'Restaurant & Entertainment',
    'Shopping & Retail',
    'Uncategorized',
    'Well-being & Lifestyle'
]

grouped_summary = df_pt_15_merged.groupby('from_id')[category_cols + ['pt_co2']].agg({
    'Educational Facilities': 'sum',
    'Grocery Stores & Supermarkets': 'sum',
    'Jobs, Professional Services & Religious': 'sum',
    'Restaurant & Entertainment': 'sum',
    'Shopping & Retail': 'sum',
    'Uncategorized': 'sum',
    'Well-being & Lifestyle': 'sum',
    'pt_co2': 'mean'
}).reset_index()



In [ ]:
# Step 6: Create total_pois column (excluding 'Uncategorized')
grouped_summary['total_pois'] = grouped_summary[[
    'Educational Facilities',
    'Grocery Stores & Supermarkets',
    'Jobs, Professional Services & Religious',
    'Restaurant & Entertainment',
    'Shopping & Retail',
    'Well-being & Lifestyle'
]].sum(axis=1)

In [ ]:
grouped_summary.head()

In [ ]:
import geopandas as gpd
import h3
from shapely.geometry import Polygon
import contextily as ctx; import basemaps
import matplotlib.pyplot as plt
import mapclassify

# Function to convert h3 to polygon
def h3_to_polygon(h):
    return Polygon(h3.h3_to_geo_boundary(h, geo_json=True))

# Convert 'from_id' to geometry
grouped_summary = grouped_summary.copy()
geometry = grouped_summary['from_id'].apply(h3_to_polygon)
gdf_hexes = gpd.GeoDataFrame(grouped_summary, geometry=geometry, crs='EPSG:4326')

# Classify total_pois into 5 natural breaks
classifier = mapclassify.NaturalBreaks(y=gdf_hexes['total_pois'], k=5)
gdf_hexes['poi_class'] = classifier.yb

# Plot
fig, ax = plt.subplots(figsize=(10, 10))
gdf_hexes.plot(ax=ax, column='poi_class', cmap='viridis', legend=True, edgecolor='black', linewidth=0.2)
ctx.add_basemap(ax, source=basemaps.POSITRON, crs=gdf_hexes.crs.to_string())
ax.set_title("Total POIs per Hexagon (Natural Breaks)")
ax.axis('off')
plt.tight_layout()
plt.show()



In [ ]:
#gdf_hexes[gdf_hexes["total_pois"]>=1000].explore()

In [ ]:
home_work = pd.read_parquet("data/work_home_locations_april.parquet")

In [ ]:
import h3
import shapely.geometry

# Function to convert H3 index to shapely Polygon
def h3_to_geom(h3_index):
    boundary = h3.h3_to_geo_boundary(h3_index, geo_json=True)
    return shapely.geometry.Polygon(boundary)

# Apply to home and work
home_work['home_geom'] = home_work['home_gid'].apply(h3_to_geom)
home_work['work_geom'] = home_work['final_work_gid'].apply(h3_to_geom)

# Drop the gid columns (not needed downstream)
# home_work = home_work.drop(columns=['home_gid', 'final_work_gid'])

# Make it a GeoDataFrame
import geopandas as gpd

home_work_gdf = gpd.GeoDataFrame(
    home_work,
    geometry='home_geom',
    crs='EPSG:4326'
)

In [ ]:
duplicates = home_work_gdf[
    home_work_gdf.duplicated(subset='home_geom', keep=False)
]


In [ ]:
data = gpd.read_file("data/finland_mun.geojson")

data_helsinki = data[data["NAMEFIN"]=="Helsinki"]

# wgs crs to geodataframe
helsinki_geo = data_helsinki.to_crs(4326)

In [ ]:
homes_in_helsinki = gpd.sjoin(
    home_work_gdf,
    helsinki_geo,
    how='inner',
    predicate='intersects'
)


In [ ]:
homes_in_helsinki.shape

In [ ]:
home_work_gdf = home_work_gdf[home_work_gdf["home_total_weight"] >= 120]

In [ ]:
home_work_gdf

In [ ]:
# Step 1: Drop duplicates based on 'gid', keeping the first occurrence
unique_home_gdf = home_work_gdf.drop_duplicates(subset="home_gid", keep="first")

# Step 2: Keep only relevant columns, for example 'gid' and 'home_geom'
unique_home_gdf = unique_home_gdf[["home_gid", "home_geom"]]

# Now unique_home_gdf contains one row per unique 'gid', and the associated 'home_geom'
print(unique_home_gdf.head())

In [ ]:
unique_home_gdf